In [1]:
# Importing Libraries 
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch 
import torch.nn as nn
import torch.optim as optim 
import torch.nn.functional as F
import time 
import torchvision.models as models 
from matplotlib import pyplot as plt 
import optuna

c:\Users\pksju\Desktop\code\Car_damage_prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

### Load Data

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset_path = r"C:\Users\pksju\Desktop\code\Car_damage_prediction\dataset"

dataset = datasets.ImageFolder(dataset_path, transform=image_transforms)
len(dataset)

2296

In [5]:
num_classes = len(dataset.classes)
num_classes

6

In [6]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size
train_size, val_size

(1722, 574)

In [7]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [9]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([32, 3, 224, 224])
torch.Size([32])


### Model : RestNet ; Model Training and Hyperparameter Tunning

In [10]:
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')

        # freeze all the layers except the final Fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False

        # freeze all the layers except the final Fully connected layer
        for param in self.model.layer4.parameters():
            param.requires_grad = True


        # To update lastly FCN (Fully Connected network) ==> model.classifier
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)

        )

    def forward(self, x):
        x = self.model(x)
        return x
        

In [11]:
def objective(trial):
    # suggested value for the hyperparamters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)

    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    #Training loop (using fewer epochs for faster hyperparamter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation Loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)

        # Handle pruning 
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
    end = time.time()
    print(f"Execution time: {end - start} seconds")

    return accuracy

In [12]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')  
study.optimize(objective, n_trials=20)

[I 2026-05-06 16:46:09,005] A new study created in memory with name: no-name-995e6144-8133-44e0-9d26-9a2a9350db4f
[I 2026-05-06 17:03:02,190] Trial 0 finished with value: 71.602787456446 and parameters: {'lr': 4.558567517476869e-05, 'dropout_rate': 0.3460171116239448}. Best is trial 0 with value: 71.602787456446.


Execution time: 1012.0203337669373 seconds


[I 2026-05-06 17:20:10,592] Trial 1 finished with value: 76.30662020905923 and parameters: {'lr': 0.007855033133196179, 'dropout_rate': 0.39136387992910193}. Best is trial 1 with value: 76.30662020905923.


Execution time: 1027.8030061721802 seconds


[I 2026-05-06 17:36:45,886] Trial 2 finished with value: 73.5191637630662 and parameters: {'lr': 6.548594874039305e-05, 'dropout_rate': 0.47544354401342875}. Best is trial 1 with value: 76.30662020905923.


...

Execution time: 1115.6653740406036 seconds


[I 2026-05-06 21:07:29,656] Trial 18 finished with value: 77.87456445993031 and parameters: {'lr': 0.00037723835044632534, 'dropout_rate': 0.46637419120098306}. Best is trial 17 with value: 80.31358885017421.


Execution time: 1068.686396598816 seconds


[I 2026-05-06 21:24:31,517] Trial 19 finished with value: 78.9198606271777 and parameters: {'lr': 0.000667337583099581, 'dropout_rate': 0.2963872451337563}. Best is trial 17 with value: 80.31358885017421.


Execution time: 1021.1737804412842 seconds


### Print best parameters


In [13]:
study.best_params

{'lr': 0.0005001980102235725, 'dropout_rate': 0.4376952174255673}